# 04. Model Evaluation & Error Analysis

This notebook evaluates the best trained ISL gesture classifier on the held-out test set (`X_test.npz`), generating per-class metrics, confusion matrix heatmaps, and error logs.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from src.models.model_utils import build_model, load_checkpoint
from src.preprocessing.config import TOTAL_FEATURES, SEQ_LEN

SPLITS_DIR = 'data/splits'
CHECKPOINT_PATH = 'checkpoints/best_lstm_model.pt'
FIGURES_DIR = 'reports/figures'
METRICS_OUTPUT = 'reports/test_metrics.json'

os.makedirs(FIGURES_DIR, exist_ok=True)

In [ ]:
# Load Test Dataset & Class Mapping
test_data = np.load(os.path.join(SPLITS_DIR, 'X_test.npz'))
X_test, y_test = test_data['X'], test_data['y']

with open(os.path.join(SPLITS_DIR, 'class_index_to_label.json'), 'r') as f:
    idx_to_label = json.load(f)

class_names = [idx_to_label[str(i)] for i in range(len(idx_to_label))]
num_classes = len(class_names)
print(f"Loaded {len(X_test)} test samples across {num_classes} gesture classes.")

In [ ]:
# Load Trained Model Checkpoint
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_model('lstm', num_features=TOTAL_FEATURES, num_classes=num_classes, seq_len=SEQ_LEN)
model, _ = load_checkpoint(CHECKPOINT_PATH, model, device=device)

# Run Inference
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
with torch.no_grad():
    logits = model(X_test_tensor)
    preds = torch.argmax(logits, dim=1).cpu().numpy()

acc = accuracy_score(y_test, preds)
print(f"=== Held-out Test Accuracy: {acc * 100:.2f}% ===")

In [ ]:
# Compute Classification Report & Confusion Matrix
clf_report = classification_report(y_test, preds, target_names=class_names, output_dict=True)
print(classification_report(y_test, preds, target_names=class_names))

cm = confusion_matrix(y_test, preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('ISL Gesture Recognition — Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
cm_path = os.path.join(FIGURES_DIR, 'confusion_matrix.png')
plt.savefig(cm_path, dpi=300)
plt.show()
print(f"Saved confusion matrix heatmap to {cm_path}")

In [ ]:
# Save Test Metrics JSON
metrics_data = {
    'test_accuracy': float(acc),
    'num_test_samples': int(len(y_test)),
    'classification_report': clf_report
}
with open(METRICS_OUTPUT, 'w') as f:
    json.dump(metrics_data, f, indent=2)
print(f"Saved detailed test metrics to {METRICS_OUTPUT}")